In [ ]:
# Se importan todas las librerías necesarias
import pandas as pd
import seaborn as sns
import numpy as np  
from matplotlib import pyplot as plt
import pickle

# Importante poner para que se muestren los gráficos
%matplotlib inline

# Semilla que se usará siempre para garantizar reproducibilidad
semilla = 123

# Directorio con los ficheros
dir_datasets = "./VulkanSimTrain/"

In [ ]:
nombres_escenas = ["BATH", "BUNNY", "CAR", "CHSNT", "CRNVL", "FOX", "FRST", "LANDS", "PARK", "PARTY", "REF", "ROBOT", "SHIP", "SPNZA", "SPRNG", "WKND"]

datos = dict()

for nombre in nombres_escenas:
    print(f"Leyendo escena {nombre}")
    datos[nombre] = pd.read_csv(dir_datasets+nombre+"-frame0.txt",sep=",",header=None,names=["ID","Cycle","Address","IP","Cache_hit"])


In [ ]:
for nombre in nombres_escenas:
    print(f"Escena: {nombre}")
    print()
    print(datos[nombre].head())
    print("-"*80)

In [ ]:
for nombre in nombres_escenas:
    print(f"Escena: {nombre}")
    print()
    print(datos[nombre].dtypes)
    print("-"*80)

In [ ]:
# Se pasa la columna Address de hexadecimal a decimal
for nombre in nombres_escenas:
    print("Escena",nombre)
    print()
    datos[nombre]["Address"] = datos[nombre]["Address"].apply(lambda x: int(str(x), 16))
    print("-"*80)

In [ ]:
# Se crea la columna Block_address dividiendo los valores de la
# columna Address entre 128
for nombre in nombres_escenas:
    print("Escena",nombre)
    print()
    datos[nombre]['Block_address'] = datos[nombre]['Address'].apply(lambda x: x//128)
    print("-"*80)


In [ ]:
# Se pasa la columna IP de hexadecimal a decimal, asignando -1 a los valores ffffffffffffffff
for nombre in nombres_escenas:
    print("Escena",nombre)
    print()
    datos[nombre]['IP'] = datos[nombre]['IP'].apply(lambda x: int(x, 16) if x != "ffffffffffffffff" else -1)
    print("-"*80)

In [ ]:
# Se crea la columna Page_address dividiendo los valores de la
# columna Address entre 4096
for nombre in nombres_escenas:
    print("Escena",nombre)
    print()
    datos[nombre]['Page_address'] = datos[nombre]['Address'].apply(lambda x: x//4096)
    print("-"*80)

In [ ]:
# Se crea la columna Delta, que muestra, para cada Block_address,
# la diferencia con el Block_address anterior
# datos['Delta'] = datos['Block_address'].diff().fillna(0).astype(int)

In [ ]:
datos["ROBOT"].head()

In [ ]:
datos["ROBOT"].dtypes

In [ ]:
datos["ROBOT"].describe()

In [ ]:
# Función para mostrar un histograma de una columna
def mostrar_columna(escena:str, columna:str, limite_absoluto:int = None):
    plt.figure(figsize=(8,5))
    if limite_absoluto is not None:
        sns.histplot(data=datos[escena][abs(datos[escena].Page_address_delta) < limite_absoluto], x=columna, bins=100)
    else:
        sns.histplot(data=datos[escena], x=columna, bins=100)
    plt.title(f'Distribución de {columna} para {escena}')
    plt.xlabel(columna)
    plt.ylabel('Recuento')
    plt.ticklabel_format(style="plain")
    plt.show()

In [ ]:
for escena in nombres_escenas:
    next_page_same_counter = 0
    next_page_different_counter = 0

    k = 5

    for i in range(len(datos[escena])-k):
        if datos[escena]["Page_address"][i] == datos[escena]["Page_address"][i+k]:
            next_page_same_counter+=1
        else:
            next_page_different_counter+=1

    print(f"Datos para la escena {escena}:")
    print(f"Saltos a la misma página: {next_page_same_counter} ({100*next_page_same_counter/(next_page_same_counter+next_page_different_counter)}%)")
    print(f"Saltos a otra página: {next_page_different_counter} ({100*next_page_different_counter/(next_page_same_counter+next_page_different_counter)}%)")
    print("="*80)

In [ ]:
for escena in nombres_escenas:
    mostrar_columna(escena,"Page_address")

In [ ]:
for escena in nombres_escenas:
    num_paginas_unicas = datos[escena]["Page_address"].nunique()
    print(f'Número de páginas únicas en la escena {escena}: {num_paginas_unicas} ({100*num_paginas_unicas/datos[escena]["Page_address"].count()}%)')

In [ ]:
class entrada_ptt:
    def __init__(self, prediccion):
        self.prediccion = prediccion
        self.confianza = 1

In [ ]:
historias = [1,2,3,4,5,6,7,8,9,10]

# La clave es la historia. El valor es una lista con los
# porcentajes para cada escena
porcentaje_predicciones_correctas = dict()

for historia in historias:
    print(f"HISTORIA {historia}")
    print("="*30)

    porcentaje_predicciones_correctas[historia] = []

    for escena in nombres_escenas:
        # La PTT ahora no guarda correspondencias entre
        # páginas pasadas y la página futura. Ahora guarda
        # correspondencias entre deltas pasados y el delta futuro
        ptt = dict()

        k = 5

        predicciones_correctas = 0
        predicciones_erroneas = 0

        for i in range(historia,len(datos[escena])-k):
            
            # Estas son las páginas que se tendrán en cuenta para calcular los deltas
            paginas_historia = tuple(datos[escena]["Page_address"][i-historia:i+1])

            # Estos son los deltas entre páginas
            deltas = []

            for j in range(1,len(paginas_historia)):
                pagina_actual = paginas_historia[j]
                pagina_pasada = paginas_historia[j-1]
                delta = pagina_actual-pagina_pasada
                deltas.append(delta)
            
            # La lista de deltas se pasa a una tupla para poder usarla
            # como clave en el diccionario
            deltas = tuple(deltas)

            print("-"*30)

            print(f"Páginas de historia: {paginas_historia}")
            print(f"Deltas: {deltas}")

            if deltas in ptt:
                # Si hay coincidencia, se utiliza el delta predicho
                # que hay en la PTT
                delta_predicho = ptt[deltas]
                print(f"Entrada de la PTT: (Predicción: {ptt[deltas].prediccion}, confianza: {ptt[deltas].confianza})")
            else:
                # Si no hay coincidencia, se considera que se
                # seguirá en la misma página (delta 0)
                delta_predicho = 0
                print("No hay entrada en la PTT")

            # Se calcula el delta real
            delta_real = (datos[escena]["Page_address"][i+k] - datos[escena]["Page_address"][i])

            print(f"Página siguiente real: {datos[escena]["Page_address"][i+k]}")
            print(f"Delta real: {delta_real}")

            # Si el delta entre la página que hay 5 accesos después y la página actual es correcto,
            # se aumenta el contador de predicciones correctas. Si no, se aumenta el contador de
            # predicciones erróneas
            if delta_real == delta_predicho:
                predicciones_correctas+=1
                print("La predicción fue correcta")
            else:
                predicciones_erroneas+=1
                print("La predicción fue incorrecta")
            
            # Si la predicción viene de una entrada de la PTT y se
            # ha predicho correctamente, se aumenta la confianza
            if delta_real == delta_predicho and deltas in ptt:
                ptt[deltas].confianza += 1
                print(f"La confianza aumenta de {ptt[deltas].confianza-1} a {ptt[deltas].confianza}")
            
            # Si la predicción viene de una entrada de la PTT y se
            # ha predicho incorrectamente, se disminuye la confianza,
            # eliminando la entrada si la confianza llega a 0
            elif delta_real != delta_predicho and deltas in ptt:
                ptt[deltas].confianza -= 1
                print(f"La confianza disminuye de {ptt[deltas].confianza+1} a {ptt[deltas].confianza}")
                if ptt[deltas].confianza == 0:
                    ptt.pop(deltas)
                    print("Como la confianza ha llegado a 0, se elimina la entrada de la PTT")
            
            # Después de actualizar las confianzas puede que se haya
            # liberado una entrada por llegar a confianza 0. También
            # puede que no haya habido una entrada con esta clave
            # en primere lugar. Sea como sea, se añade la entrada si
            # no hay
            if deltas not in ptt:
                ptt[deltas] = entrada_ptt(delta_real)
                print(f"No hay entrada en la PTT con esta clave. Creando entrada: (Predicción: {ptt[deltas].prediccion}, confianza: {ptt[deltas].confianza})")

        print(f"Predicciones correctas en la escena {escena}: {predicciones_correctas} ({100*predicciones_correctas/(predicciones_correctas+predicciones_erroneas)}%)")
        porcentaje_predicciones_correctas[historia].append(100*predicciones_correctas/(predicciones_correctas+predicciones_erroneas))
    print()

In [ ]:
with open('porcentaje_predicciones_correctas_deltas_confianza_en_reemplazo.p', 'wb') as fp:
    pickle.dump(porcentaje_predicciones_correctas, fp, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
with open('porcentaje_predicciones_correctas_deltas_confianza_en_reemplazo.p', 'rb') as fp:
    porcentaje_predicciones_correctas = pickle.load(fp)

In [ ]:
# Source - https://stackoverflow.com/a
# Posted by willeM_ Van Onsem, modified by community. See post 'Timeline' for change history
# Retrieved 2026-01-19, License - CC BY-SA 4.0

import numpy as np

def geo_mean(iterable):
    a = np.array(iterable)
    return a.prod()**(1.0/len(a))

In [ ]:
for h in porcentaje_predicciones_correctas.keys():
    media_geometrica = geo_mean(porcentaje_predicciones_correctas[h])
    porcentaje_predicciones_correctas[h].append(media_geometrica)
    print(f"Media geométrica con historia {h}: {media_geometrica}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

nombres_escenas = ["BATH", "BUNNY", "CAR", "CHSNT", "CRNVL", "FOX", "FRST", "LANDS", "PARK", "PARTY", "REF", "ROBOT", "SHIP", "SPNZA", "SPRNG", "WKND", "Media Geométrica"]

# Localización de cada etiqueta
x = np.arange(len(nombres_escenas))

# Anchura de las barras
width = 0.08

multiplier = 0

fig, ax = plt.subplots(layout='constrained')

fig.set_figwidth(18)

for historia, medidas in porcentaje_predicciones_correctas.items():
    offset = width * multiplier
    rects = ax.bar(x + offset, medidas, width, label=f"Historia {historia}")
    #ax.bar_label(rects, padding=3)
    multiplier += 1

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('Porcentaje de aciertos en la predicción')
ax.set_title('Porcentaje de aciertos de la PTT según la longitud de la historia, usando confianza en el reemplazo')
ax.set_xticks(x + width*4.5, nombres_escenas)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05),
          fancybox=True, shadow=True, ncol=5)

plt.show()